In [0]:
%run 
"./00setup_config"

Connected. Database ready.


path,name,size,modificationTime
abfss://landing@medallionhealthdata03.dfs.core.windows.net/appointments.csv,appointments.csv,10907,1783938174000
abfss://landing@medallionhealthdata03.dfs.core.windows.net/billing.csv,billing.csv,10018,1783938174000
abfss://landing@medallionhealthdata03.dfs.core.windows.net/doctors.csv,doctors.csv,962,1783938174000
abfss://landing@medallionhealthdata03.dfs.core.windows.net/patients.csv,patients.csv,5626,1783938174000
abfss://landing@medallionhealthdata03.dfs.core.windows.net/treatments.csv,treatments.csv,11072,1783938174000


Batch ID : 3f6a3ded-77bf-47b4-adb2-e511534929d7
Pipeline Version : 1.0
Run Time : 2026-07-13 10:52:12.878727


# Setting metadata

source_name,file_name,target_table,primary_key,load_type,active_flag
patients,patients.csv,bronze_patients,Patient_ID,FULL,Y
doctors,doctors.csv,bronze_doctors,Doctor_ID,FULL,Y
appointments,appointments.csv,bronze_appointments,Appointment_ID,FULL,Y
billing,billing.csv,bronze_billing,Billing_ID,FULL,Y
treatments,treatments.csv,bronze_treatments,Treatment_ID,FULL,Y


source_name,file_name,target_table,primary_key,load_type,active_flag
patients,patients.csv,bronze_patients,Patient_ID,FULL,Y
doctors,doctors.csv,bronze_doctors,Doctor_ID,FULL,Y
appointments,appointments.csv,bronze_appointments,Appointment_ID,FULL,Y
billing,billing.csv,bronze_billing,Billing_ID,FULL,Y
treatments,treatments.csv,bronze_treatments,Treatment_ID,FULL,Y


batch_id,source_name,layer,rows_read,rows_written,status,start_time,end_time,error_message


In [0]:

from pyspark.sql import functions as F
from datetime import datetime
import uuid

In [0]:
metadata_df = (
    spark.read
    .format("delta")
    .load(f"{bronze_path}metadata_config")
)

In [0]:
active_sources = (
    metadata_df
    .filter(F.col("active_flag") == "Y")
    .collect()
)

In [0]:
patients_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{landing_path}patients.csv")
)

In [0]:
display(patients_df.limit(10))

patient_id,first_name,last_name,gender,date_of_birth,contact_number,address,registration_date,insurance_provider,insurance_number,email
P001,David,Williams,F,1955-06-04,6939585183,789 Pine Rd,2022-06-23,WellnessCorp,INS840674,david.williams@mail.com
P002,Emily,Smith,F,1984-10-12,8228188767,321 Maple Dr,2022-01-15,PulseSecure,INS354079,emily.smith@mail.com
P003,Laura,Jones,M,1977-08-21,8397029847,321 Maple Dr,2022-02-07,PulseSecure,INS650929,laura.jones@mail.com
P004,Michael,Johnson,F,1981-02-20,9019443432,123 Elm St,2021-03-02,HealthIndia,INS789944,michael.johnson@mail.com
P005,David,Wilson,M,1960-06-23,7734463155,123 Elm St,2021-09-29,MedCare Plus,INS788105,david.wilson@mail.com
P006,Linda,Jones,M,1963-06-16,7561777264,321 Maple Dr,2022-10-02,HealthIndia,INS613758,linda.jones@mail.com
P007,Alex,Johnson,F,1989-06-08,6278710077,789 Pine Rd,2021-12-25,MedCare Plus,INS465890,alex.johnson@mail.com
P008,David,Davis,F,1976-07-05,7090558393,456 Oak Ave,2021-05-25,WellnessCorp,INS545101,david.davis@mail.com
P009,Laura,Davis,M,1971-12-11,7060324619,321 Maple Dr,2022-09-18,PulseSecure,INS136631,laura.davis@mail.com
P010,Michael,Taylor,M,2001-10-13,7081396733,123 Elm St,2022-08-24,WellnessCorp,INS866577,michael.taylor@mail.com


In [0]:
patients_df.printSchema()

root
 |-- patient_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- contact_number: long (nullable = true)
 |-- address: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- insurance_provider: string (nullable = true)
 |-- insurance_number: string (nullable = true)
 |-- email: string (nullable = true)



In [0]:

print("Patient Records:", patients_df.count())
     

Patient Records: 50


In [0]:
patients_bronze = (
    patients_df
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_file_name", F.lit("patients.csv"))
    .withColumn("_batch_id", F.lit(batch_id))
    .withColumn("_layer", F.lit("BRONZE"))
    .withColumn("_pipeline_version", F.lit(pipeline_version))
    .withColumn("_ingestion_date", F.current_date())
)

In [0]:
display(patients_bronze)

patient_id,first_name,last_name,gender,date_of_birth,contact_number,address,registration_date,insurance_provider,insurance_number,email,_ingestion_timestamp,_source_file_name,_batch_id,_layer,_pipeline_version,_ingestion_date
P001,David,Williams,F,1955-06-04,6939585183,789 Pine Rd,2022-06-23,WellnessCorp,INS840674,david.williams@mail.com,2026-07-13T10:52:28.656306Z,patients.csv,3f6a3ded-77bf-47b4-adb2-e511534929d7,BRONZE,1.0,2026-07-13
P002,Emily,Smith,F,1984-10-12,8228188767,321 Maple Dr,2022-01-15,PulseSecure,INS354079,emily.smith@mail.com,2026-07-13T10:52:28.656306Z,patients.csv,3f6a3ded-77bf-47b4-adb2-e511534929d7,BRONZE,1.0,2026-07-13
P003,Laura,Jones,M,1977-08-21,8397029847,321 Maple Dr,2022-02-07,PulseSecure,INS650929,laura.jones@mail.com,2026-07-13T10:52:28.656306Z,patients.csv,3f6a3ded-77bf-47b4-adb2-e511534929d7,BRONZE,1.0,2026-07-13
P004,Michael,Johnson,F,1981-02-20,9019443432,123 Elm St,2021-03-02,HealthIndia,INS789944,michael.johnson@mail.com,2026-07-13T10:52:28.656306Z,patients.csv,3f6a3ded-77bf-47b4-adb2-e511534929d7,BRONZE,1.0,2026-07-13
P005,David,Wilson,M,1960-06-23,7734463155,123 Elm St,2021-09-29,MedCare Plus,INS788105,david.wilson@mail.com,2026-07-13T10:52:28.656306Z,patients.csv,3f6a3ded-77bf-47b4-adb2-e511534929d7,BRONZE,1.0,2026-07-13
P006,Linda,Jones,M,1963-06-16,7561777264,321 Maple Dr,2022-10-02,HealthIndia,INS613758,linda.jones@mail.com,2026-07-13T10:52:28.656306Z,patients.csv,3f6a3ded-77bf-47b4-adb2-e511534929d7,BRONZE,1.0,2026-07-13
P007,Alex,Johnson,F,1989-06-08,6278710077,789 Pine Rd,2021-12-25,MedCare Plus,INS465890,alex.johnson@mail.com,2026-07-13T10:52:28.656306Z,patients.csv,3f6a3ded-77bf-47b4-adb2-e511534929d7,BRONZE,1.0,2026-07-13
P008,David,Davis,F,1976-07-05,7090558393,456 Oak Ave,2021-05-25,WellnessCorp,INS545101,david.davis@mail.com,2026-07-13T10:52:28.656306Z,patients.csv,3f6a3ded-77bf-47b4-adb2-e511534929d7,BRONZE,1.0,2026-07-13
P009,Laura,Davis,M,1971-12-11,7060324619,321 Maple Dr,2022-09-18,PulseSecure,INS136631,laura.davis@mail.com,2026-07-13T10:52:28.656306Z,patients.csv,3f6a3ded-77bf-47b4-adb2-e511534929d7,BRONZE,1.0,2026-07-13
P010,Michael,Taylor,M,2001-10-13,7081396733,123 Elm St,2022-08-24,WellnessCorp,INS866577,michael.taylor@mail.com,2026-07-13T10:52:28.656306Z,patients.csv,3f6a3ded-77bf-47b4-adb2-e511534929d7,BRONZE,1.0,2026-07-13


In [0]:
patients_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .save(f"{bronze_path}patients")

In [0]:
bronze_patients = (
    spark.read
    .format("delta")
    .load(f"{bronze_path}patients")
)

display(bronze_patients)

patient_id,first_name,last_name,gender,date_of_birth,contact_number,address,registration_date,insurance_provider,insurance_number,email,_ingestion_timestamp,_source_file_name,_batch_id,_layer,_pipeline_version,_ingestion_date
P001,David,Williams,F,1955-06-04,6939585183,789 Pine Rd,2022-06-23,WellnessCorp,INS840674,david.williams@mail.com,2026-07-13T10:52:30.198792Z,patients.csv,3f6a3ded-77bf-47b4-adb2-e511534929d7,BRONZE,1.0,2026-07-13
P002,Emily,Smith,F,1984-10-12,8228188767,321 Maple Dr,2022-01-15,PulseSecure,INS354079,emily.smith@mail.com,2026-07-13T10:52:30.198792Z,patients.csv,3f6a3ded-77bf-47b4-adb2-e511534929d7,BRONZE,1.0,2026-07-13
P003,Laura,Jones,M,1977-08-21,8397029847,321 Maple Dr,2022-02-07,PulseSecure,INS650929,laura.jones@mail.com,2026-07-13T10:52:30.198792Z,patients.csv,3f6a3ded-77bf-47b4-adb2-e511534929d7,BRONZE,1.0,2026-07-13
P004,Michael,Johnson,F,1981-02-20,9019443432,123 Elm St,2021-03-02,HealthIndia,INS789944,michael.johnson@mail.com,2026-07-13T10:52:30.198792Z,patients.csv,3f6a3ded-77bf-47b4-adb2-e511534929d7,BRONZE,1.0,2026-07-13
P005,David,Wilson,M,1960-06-23,7734463155,123 Elm St,2021-09-29,MedCare Plus,INS788105,david.wilson@mail.com,2026-07-13T10:52:30.198792Z,patients.csv,3f6a3ded-77bf-47b4-adb2-e511534929d7,BRONZE,1.0,2026-07-13
P006,Linda,Jones,M,1963-06-16,7561777264,321 Maple Dr,2022-10-02,HealthIndia,INS613758,linda.jones@mail.com,2026-07-13T10:52:30.198792Z,patients.csv,3f6a3ded-77bf-47b4-adb2-e511534929d7,BRONZE,1.0,2026-07-13
P007,Alex,Johnson,F,1989-06-08,6278710077,789 Pine Rd,2021-12-25,MedCare Plus,INS465890,alex.johnson@mail.com,2026-07-13T10:52:30.198792Z,patients.csv,3f6a3ded-77bf-47b4-adb2-e511534929d7,BRONZE,1.0,2026-07-13
P008,David,Davis,F,1976-07-05,7090558393,456 Oak Ave,2021-05-25,WellnessCorp,INS545101,david.davis@mail.com,2026-07-13T10:52:30.198792Z,patients.csv,3f6a3ded-77bf-47b4-adb2-e511534929d7,BRONZE,1.0,2026-07-13
P009,Laura,Davis,M,1971-12-11,7060324619,321 Maple Dr,2022-09-18,PulseSecure,INS136631,laura.davis@mail.com,2026-07-13T10:52:30.198792Z,patients.csv,3f6a3ded-77bf-47b4-adb2-e511534929d7,BRONZE,1.0,2026-07-13
P010,Michael,Taylor,M,2001-10-13,7081396733,123 Elm St,2022-08-24,WellnessCorp,INS866577,michael.taylor@mail.com,2026-07-13T10:52:30.198792Z,patients.csv,3f6a3ded-77bf-47b4-adb2-e511534929d7,BRONZE,1.0,2026-07-13


In [0]:
def load_to_bronze(file_name):
    
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"{landing_path}{file_name}.csv")
    )

    bronze_df = (
        df
        .withColumn("_ingestion_timestamp", F.current_timestamp())
        .withColumn("_source_file_name", F.lit(f"{file_name}.csv"))
        .withColumn("_batch_id", F.lit(batch_id))
        .withColumn("_layer", F.lit("BRONZE"))
        .withColumn("_pipeline_version", F.lit(pipeline_version))
        .withColumn("_ingestion_date", F.current_date())
    )

    (
        bronze_df.write
        .format("delta")
        .mode("overwrite")
        .save(f"{bronze_path}{file_name}")
    )

    print(f"✅ {file_name} loaded successfully.")

In [0]:
files = [
    "patients",
    "doctors",
    "appointments",
    "billing",
    "treatments"
]

for file in files:
    load_to_bronze(file)

✅ patients loaded successfully.
✅ doctors loaded successfully.
✅ appointments loaded successfully.
✅ billing loaded successfully.
✅ treatments loaded successfully.


In [0]:
display(dbutils.fs.ls(bronze_path))

path,name,size,modificationTime
abfss://bronze@medallionhealthdata03.dfs.core.windows.net/appointments/,appointments/,0,1783939020000
abfss://bronze@medallionhealthdata03.dfs.core.windows.net/audit_log/,audit_log/,0,1783938655000
abfss://bronze@medallionhealthdata03.dfs.core.windows.net/billing/,billing/,0,1783939025000
abfss://bronze@medallionhealthdata03.dfs.core.windows.net/doctors/,doctors/,0,1783939015000
abfss://bronze@medallionhealthdata03.dfs.core.windows.net/metadata_config/,metadata_config/,0,1783938588000
abfss://bronze@medallionhealthdata03.dfs.core.windows.net/patients/,patients/,0,1783938981000
abfss://bronze@medallionhealthdata03.dfs.core.windows.net/treatments/,treatments/,0,1783939029000
